<a href="https://colab.research.google.com/github/OmegaS48/Quant_Personal/blob/main/QA_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Deutsch Algorithm implementation using CSV-defined Oracles
# qiskit version 1.4.5 — IBM Quantum Runtime version
import pandas as pd
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# ── IBM Credentials ───────────────────────────────────────────────────────────
IBM_API_TOKEN = "rGWg1efAWuQ03posXiiGqV7KOLMzx_0y56WJmK_Tte88"
IBM_INSTANCE  = "crn:v1:bluemix:public:quantum-computing:us-east:a/b15ddaf5b835425b864ed38166696266:d16b1393-6852-4c65-8eea-9e725a967d58::"

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_API_TOKEN, instance=IBM_INSTANCE)
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on: {backend.name}")
# ─────────────────────────────────────────────────────────────────────────────


def load_oracle_from_csv(file_path):
    df = pd.read_csv(file_path, dtype=str).fillna("")
    num_qubits = 1
    oracle_qc = QuantumCircuit(num_qubits + 1)

    outputs = df["f(x)"].astype(str).str.strip()
    ones_count = (outputs == "1").sum()
    total_possible_inputs = 2**num_qubits

    if ones_count == 0:
        print("Optimization: Detected Constant 0. Oracle remains Identity.")
        return oracle_qc, num_qubits

    if ones_count == total_possible_inputs:
        print("Optimization: Detected Constant 1. Applying single X gate to ancilla.")
        oracle_qc.x(num_qubits)
        return oracle_qc, num_qubits

    for index, row in df.iterrows():
        output_val = str(row["f(x)"]).strip()
        input_str = str(row["x"]).strip()

        if output_val == "1":
            bit_string = input_str.zfill(num_qubits)
            print(f"Adding Oracle Logic for f({bit_string}) = 1")

            for i, bit in enumerate(reversed(bit_string)):
                if bit == "0":
                    oracle_qc.x(i)

            oracle_qc.mcx(list(range(num_qubits)), num_qubits)

            for i, bit in enumerate(reversed(bit_string)):
                if bit == "0":
                    oracle_qc.x(i)

            oracle_qc.barrier()

    return oracle_qc, num_qubits


def run_deutsch_experiment(file_path, shots):
    oracle_f, n = load_oracle_from_csv(file_path)

    if n != 1:
        print(f"Warning: Deutsch Algorithm usually uses 1 input qubit. CSV defines {n}.")

    deutsch_circuit = QuantumCircuit(n + 1, n)
    deutsch_circuit.x(n)
    deutsch_circuit.h(range(n + 1))
    deutsch_circuit.barrier()
    deutsch_circuit.compose(oracle_f, inplace=True)
    deutsch_circuit.barrier()
    deutsch_circuit.h(range(n))
    deutsch_circuit.measure(range(n), range(n))

    # Transpile and run on IBM
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    isa_circuit = pm.run(deutsch_circuit)
    job = Sampler(backend).run([isa_circuit], shots=shots)
    print(f"Job ID: {job.job_id()} | Waiting for results...")
    result = job.result()
    counts = result[0].data.c.get_counts()

    zero_count = counts.get("0", 0)
    prob_zero = (zero_count / shots) * 100

    print("-" * 30)
    print(f"DEUTSCH RESULTS (1 Input Qubit)")
    print("-" * 30)
    print(f"Measurement Counts: {counts}")

    if prob_zero > 99:
        print("Conclusion: Function is CONSTANT")
    elif prob_zero < 1:
        print("Conclusion: Function is BALANCED")
    else:
        print(f"Conclusion: Non-standard function (noise may have affected results).")

    print("\nORACLE:")
    print(oracle_f.draw(output="text"))
    print("\nFULL DEUTSCH CIRCUIT:")
    print(deutsch_circuit.draw(output="text", fold=-1))


# --- Execution ---
shots = int(input("Enter the number of shots: "))
try:
    run_deutsch_experiment("1_function_design.csv", shots)
except Exception as e:
    print(f"Error: {e}. Ensure 'function_design.csv' is correct.")

ModuleNotFoundError: No module named 'qiskit'

In [2]:
!pip install qiskit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.6 MB/s eta 0:00:00


In [3]:
# Deutsch Algorithm implementation using CSV-defined Oracles
# qiskit version 1.4.5 — IBM Quantum Runtime version
import pandas as pd
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# ── IBM Credentials ───────────────────────────────────────────────────────────
IBM_API_TOKEN = "rGWg1efAWuQ03posXiiGqV7KOLMzx_0y56WJmK_Tte88"
IBM_INSTANCE  = "crn:v1:bluemix:public:quantum-computing:us-east:a/b15ddaf5b835425b864ed38166696266:d16b1393-6852-4c65-8eea-9e725a967d58::"

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_API_TOKEN, instance=IBM_INSTANCE)
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on: {backend.name}")
# ─────────────────────────────────────────────────────────────────────────────


def load_oracle_from_csv(file_path):
    df = pd.read_csv(file_path, dtype=str).fillna("")
    num_qubits = 1
    oracle_qc = QuantumCircuit(num_qubits + 1)

    outputs = df["f(x)"].astype(str).str.strip()
    ones_count = (outputs == "1").sum()
    total_possible_inputs = 2**num_qubits

    if ones_count == 0:
        print("Optimization: Detected Constant 0. Oracle remains Identity.")
        return oracle_qc, num_qubits

    if ones_count == total_possible_inputs:
        print("Optimization: Detected Constant 1. Applying single X gate to ancilla.")
        oracle_qc.x(num_qubits)
        return oracle_qc, num_qubits

    for index, row in df.iterrows():
        output_val = str(row["f(x)"]).strip()
        input_str = str(row["x"]).strip()

        if output_val == "1":
            bit_string = input_str.zfill(num_qubits)
            print(f"Adding Oracle Logic for f({bit_string}) = 1")

            for i, bit in enumerate(reversed(bit_string)):
                if bit == "0":
                    oracle_qc.x(i)

            oracle_qc.mcx(list(range(num_qubits)), num_qubits)

            for i, bit in enumerate(reversed(bit_string)):
                if bit == "0":
                    oracle_qc.x(i)

            oracle_qc.barrier()

    return oracle_qc, num_qubits


def run_deutsch_experiment(file_path, shots):
    oracle_f, n = load_oracle_from_csv(file_path)

    if n != 1:
        print(f"Warning: Deutsch Algorithm usually uses 1 input qubit. CSV defines {n}.")

    deutsch_circuit = QuantumCircuit(n + 1, n)
    deutsch_circuit.x(n)
    deutsch_circuit.h(range(n + 1))
    deutsch_circuit.barrier()
    deutsch_circuit.compose(oracle_f, inplace=True)
    deutsch_circuit.barrier()
    deutsch_circuit.h(range(n))
    deutsch_circuit.measure(range(n), range(n))

    # Transpile and run on IBM
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    isa_circuit = pm.run(deutsch_circuit)
    job = Sampler(backend).run([isa_circuit], shots=shots)
    print(f"Job ID: {job.job_id()} | Waiting for results...")
    result = job.result()
    counts = result[0].data.c.get_counts()

    zero_count = counts.get("0", 0)
    prob_zero = (zero_count / shots) * 100

    print("-" * 30)
    print(f"DEUTSCH RESULTS (1 Input Qubit)")
    print("-" * 30)
    print(f"Measurement Counts: {counts}")

    if prob_zero > 99:
        print("Conclusion: Function is CONSTANT")
    elif prob_zero < 1:
        print("Conclusion: Function is BALANCED")
    else:
        print(f"Conclusion: Non-standard function (noise may have affected results).")

    print("\nORACLE:")
    print(oracle_f.draw(output="text"))
    print("\nFULL DEUTSCH CIRCUIT:")
    print(deutsch_circuit.draw(output="text", fold=-1))


# --- Execution ---
shots = int(input("Enter the number of shots: "))
try:
    run_deutsch_experiment("1_function_design.csv", shots)
except Exception as e:
    print(f"Error: {e}. Ensure 'function_design.csv' is correct.")

ModuleNotFoundError: No module named 'qiskit_ibm_runtime'

In [4]:
!pip install qiskit_ibm_runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.8/381.8 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.3/205.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.8 MB/s eta 0:00:00


In [8]:
# Deutsch Algorithm implementation using CSV-defined Oracles
# qiskit version 1.4.5 — IBM Quantum Runtime version
import pandas as pd
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# ── IBM Credentials ───────────────────────────────────────────────────────────
IBM_API_TOKEN = "rGWg1efAWuQ03posXiiGqV7KOLMzx_0y56WJmK_Tte88"
IBM_INSTANCE  = "crn:v1:bluemix:public:quantum-computing:us-east:a/b090e7df031c4bdc96d7bc096d28948f:8be3a8fa-4c0a-42c5-bc9f-873cac6bbb74::"

service = QiskitRuntimeService(channel="ibm_quantum_platform", token=IBM_API_TOKEN, instance=IBM_INSTANCE)
backend = service.least_busy(operational=True, simulator=False)
print(f"Running on: {backend.name}")
# ─────────────────────────────────────────────────────────────────────────────


def load_oracle_from_csv(file_path):
    df = pd.read_csv(file_path, dtype=str).fillna("")
    num_qubits = 1
    oracle_qc = QuantumCircuit(num_qubits + 1)

    outputs = df["f(x)"].astype(str).str.strip()
    ones_count = (outputs == "1").sum()
    total_possible_inputs = 2**num_qubits

    if ones_count == 0:
        print("Optimization: Detected Constant 0. Oracle remains Identity.")
        return oracle_qc, num_qubits

    if ones_count == total_possible_inputs:
        print("Optimization: Detected Constant 1. Applying single X gate to ancilla.")
        oracle_qc.x(num_qubits)
        return oracle_qc, num_qubits

    for index, row in df.iterrows():
        output_val = str(row["f(x)"]).strip()
        input_str = str(row["x"]).strip()

        if output_val == "1":
            bit_string = input_str.zfill(num_qubits)
            print(f"Adding Oracle Logic for f({bit_string}) = 1")

            for i, bit in enumerate(reversed(bit_string)):
                if bit == "0":
                    oracle_qc.x(i)

            oracle_qc.mcx(list(range(num_qubits)), num_qubits)

            for i, bit in enumerate(reversed(bit_string)):
                if bit == "0":
                    oracle_qc.x(i)

            oracle_qc.barrier()

    return oracle_qc, num_qubits


def run_deutsch_experiment(file_path, shots):
    oracle_f, n = load_oracle_from_csv(file_path)

    if n != 1:
        print(f"Warning: Deutsch Algorithm usually uses 1 input qubit. CSV defines {n}.")

    deutsch_circuit = QuantumCircuit(n + 1, n)
    deutsch_circuit.x(n)
    deutsch_circuit.h(range(n + 1))
    deutsch_circuit.barrier()
    deutsch_circuit.compose(oracle_f, inplace=True)
    deutsch_circuit.barrier()
    deutsch_circuit.h(range(n))
    deutsch_circuit.measure(range(n), range(n))

    # Transpile and run on IBM
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    isa_circuit = pm.run(deutsch_circuit)
    job = Sampler(backend).run([isa_circuit], shots=shots)
    print(f"Job ID: {job.job_id()} | Waiting for results...")
    result = job.result()
    counts = result[0].data.c.get_counts()

    zero_count = counts.get("0", 0)
    prob_zero = (zero_count / shots) * 100

    print("-" * 30)
    print(f"DEUTSCH RESULTS (1 Input Qubit)")
    print("-" * 30)
    print(f"Measurement Counts: {counts}")

    if prob_zero > 99:
        print("Conclusion: Function is CONSTANT")
    elif prob_zero < 1:
        print("Conclusion: Function is BALANCED")
    else:
        print(f"Conclusion: Non-standard function (noise may have affected results).")

    print("\nORACLE:")
    print(oracle_f.draw(output="text"))
    print("\nFULL DEUTSCH CIRCUIT:")
    print(deutsch_circuit.draw(output="text", fold=-1))


# --- Execution ---
shots = int(input("Enter the number of shots: "))
try:
    run_deutsch_experiment("1_function_design.csv", shots)
except Exception as e:
    print(f"Error: {e}. Ensure 'function_design.csv' is correct.")

qiskit_runtime_service._discover_account:WARNING:2026-03-28 20:33:06,171: Loading account with the given token. A saved account will not be used.


Running on: ibm_fez
Enter the number of shots: 10000
Optimization: Detected Constant 1. Applying single X gate to ancilla.
Job ID: d743m2tkoquc73e2ajsg | Waiting for results...
------------------------------
DEUTSCH RESULTS (1 Input Qubit)
------------------------------
Measurement Counts: {'0': 9902, '1': 98}
Conclusion: Function is CONSTANT

ORACLE:
          
q_0: ─────
     ┌───┐
q_1: ┤ X ├
     └───┘

FULL DEUTSCH CIRCUIT:
     ┌───┐      ░       ░ ┌───┐┌─┐
q_0: ┤ H ├──────░───────░─┤ H ├┤M├
     ├───┤┌───┐ ░ ┌───┐ ░ └───┘└╥┘
q_1: ┤ X ├┤ H ├─░─┤ X ├─░───────╫─
     └───┘└───┘ ░ └───┘ ░       ║ 
c: 1/═══════════════════════════╩═
                                0 
